## 1. Download Games


In [ ]:
%pip install -U datasets

In [24]:
from huggingface_hub import login
from hfid import getHFID

# Huggingface login
login(getHFID())

In [5]:
from datasets import load_dataset

# Load a small fragment of Lichess dataset
ds = load_dataset(
    "parquet", split="train", streaming=True,
    data_files="hf://datasets/Lichess/standard-chess-games@de4e636eddf568a9394cc01fb0b9e1da04a6babf/data/year=2025/month=09/train-00017-of-00062.parquet",
)

In [6]:
# Filter for games with evaluation
ds = ds.filter(lambda x: "[%eval" in x["movetext"])
# Randomize
ds = ds.shuffle(seed=1024, buffer_size=10000)

# Take 1200 games
N_games = 1200
games = list(ds.take(N_games))

In [7]:
print(len(games))
print(games[1])

1200
{'Event': 'Rated Blitz game', 'Site': 'https://lichess.org/ih1OesOY', 'White': 'Manfred1050', 'Black': 'Amrsaadawy', 'Result': '1-0', 'WhiteTitle': None, 'BlackTitle': None, 'WhiteElo': 1420, 'BlackElo': 1427, 'WhiteRatingDiff': 6, 'BlackRatingDiff': -6, 'UTCDate': datetime.date(2025, 9, 9), 'UTCTime': datetime.time(7, 59, 41), 'ECO': 'C44', 'Opening': 'Scotch Game: Scotch Gambit, Kingside Variation', 'Termination': 'Normal', 'TimeControl': '300+3', 'movetext': '1. e4 { [%eval 0.18] [%clk 0:05:00] } 1... e5 { [%eval 0.22] [%clk 0:05:00] } 2. Nf3 { [%eval 0.13] [%clk 0:05:02] } 2... Nc6 { [%eval 0.21] [%clk 0:05:03] } 3. Bc4 { [%eval 0.12] [%clk 0:05:03] } 3... Nf6 { [%eval 0.1] [%clk 0:05:05] } 4. d4 { [%eval 0.0] [%clk 0:05:04] } 4... exd4 { [%eval 0.0] [%clk 0:05:06] } 5. e5 { [%eval 0.0] [%clk 0:05:05] } 5... Ng4 { [%eval 0.15] [%clk 0:04:57] } 6. O-O { [%eval 0.11] [%clk 0:05:04] } 6... Ngxe5? { [%eval 2.1] [%clk 0:04:54] } 7. Nxe5 { [%eval 2.67] [%clk 0:05:05] } 7... Nxe5 { [

# 2. Processing Games

In [ ]:
%pip install -U python-chess

import chess.pgn

In [9]:
from io import StringIO
# Sample game analysis
game = chess.pgn.read_game(StringIO(games[1]["movetext"]))
mateScore = 100000 # Score for mate

node = game

while node.variations:
    node = node.variation(0)
    turn = node.parent.board().turn  # Who just made that move

    print("White:" if turn else "Black:", node.parent.board().san(node.move)) # The move in SAN
    print(node.ply()) # The ply
    print(node.move.uci()) # The move
    print(node.eval().white().score(mate_score=mateScore))   # Position eval in centipawns
    print(node.eval().wdl(model="sf18", ply=node.ply()).pov(turn).expectation()) # Expected score
    print(node.clock())  # seconds remaining
    print()
    

White: e4
1
e2e4
18
0.5205
300.0

Black: e5
2
e7e5
22
0.4745
300.0

White: Nf3
3
g1f3
13
0.5145
302.0

Black: Nc6
4
b8c6
21
0.4755
303.0

White: Bc4
5
f1c4
12
0.5135
303.0

Black: Nf6
6
g8f6
10
0.4885
305.0

White: d4
7
d2d4
0
0.5
304.0

Black: exd4
8
e5d4
0
0.5
306.0

White: e5
9
e4e5
0
0.5
305.0

Black: Ng4
10
f6g4
15
0.483
297.0

White: O-O
11
e1g1
11
0.512
304.0

Black: Ngxe5
12
g4e5
210
0.0105
294.0

White: Nxe5
13
f3e5
267
0.9985
305.0

Black: Nxe5
14
c6e5
214
0.009
295.0

White: Re1
15
f1e1
221
0.993
303.0

Black: d5
16
d7d5
406
0.0
285.0

White: Rxe5+
17
e1e5
418
1.0
300.0

Black: Be6
18
c8e6
430
0.0
285.0

White: Bxd5
19
c4d5
409
1.0
294.0

Black: c6
20
c7c6
529
0.0
282.0

White: Bxe6
21
d5e6
578
1.0
291.0

Black: fxe6
22
f7e6
582
0.0
283.0

White: Rxe6+
23
e5e6
555
1.0
288.0

Black: Be7
24
f8e7
626
0.0
285.0

White: Qh5+
25
d1h5
567
1.0
277.0

Black: g6
26
g7g6
545
0.0
284.0

White: Rxg6
27
e6g6
503
1.0
274.0



In [10]:
# Compute the maximum eval and expected score for the first move, though it really is just that of 1. e4
initEval = 0 # Eval of initial position
initExp = 0 # Expected Score of initial position
for g in games:
    game = chess.pgn.read_game(StringIO(g["movetext"]))
    node = game.variation(0)
    turn = node.parent.board().turn
    initEval = max(initEval, node.eval().white().score(mate_score=mateScore))
    initExp = max(initExp, node.eval().wdl(model="sf18", ply=node.ply()).pov(turn).expectation())
print(initEval)
print(initExp)

18
0.5205


In [11]:
# Convert a game to a pandas DataFrame
import pandas as pd

blunderThreshold = 0.1 # Threshold for blunder detection

def pgn_to_df(pgn: str) -> pd.DataFrame:
    game = chess.pgn.read_game(StringIO(pgn))
    rows = []

    node = game
    prevEval = initEval
    prevExp = initExp

    while node.variations:
        node = node.variation(0)
        turn = node.parent.board().turn

        if node.eval() is None: # In most cases this is the last ply with a mate
            ev = mateScore
            exp = 1.0
        else:
            ev = node.eval().white().score(mate_score=mateScore)
            exp = node.eval().wdl(model="sf18", ply=node.ply()).pov(turn).expectation()

        rows.append({
            "ply": node.ply(),
            "move": node.move.uci(),
            "san": node.parent.board().san(node.move),
            "turn": "white" if node.parent.board().turn else "black",
            "eval": ev,
            "ev_loss": abs(prevEval - ev),
            "expScore": exp,
            "exp_loss": max(0, prevExp - exp), # Round loss up to 0 if negative
            "clock": node.clock(),
            "blunder": prevExp - exp > blunderThreshold # Blunder if expected score drops by more than the threshold
            # "fen": node.board().fen(),
        })

        if node.variations:
            prevEval = ev
            prevExp = node.eval().wdl(model="sf18", ply=node.ply()+1).pov(not turn).expectation() # Next turn ply +=1; and switch player

    return pd.DataFrame(rows)

In [12]:
print(pgn_to_df(games[1]["movetext"]))

    ply  move    san   turn  eval  ev_loss  expScore  exp_loss  clock  blunder
0     1  e2e4     e4  white    18        0    0.5205    0.0000  300.0    False
1     2  e7e5     e5  black    22        4    0.4745    0.0050  300.0    False
2     3  g1f3    Nf3  white    13        9    0.5145    0.0110  302.0    False
3     4  b8c6    Nc6  black    21        8    0.4755    0.0100  303.0    False
4     5  f1c4    Bc4  white    12        9    0.5135    0.0110  303.0    False
5     6  g8f6    Nf6  black    10        2    0.4885    0.0000  305.0    False
6     7  d2d4     d4  white     0       10    0.5000    0.0115  304.0    False
7     8  e5d4   exd4  black     0        0    0.5000    0.0000  306.0    False
8     9  e4e5     e5  white     0        0    0.5000    0.0000  305.0    False
9    10  f6g4    Ng4  black    15       15    0.4830    0.0170  297.0    False
10   11  e1g1    O-O  white    11        4    0.5120    0.0050  304.0    False
11   12  g4e5  Ngxe5  black   210      199    0.0105

In [20]:
data = []
n = 0
N_games = 1000 # We just keep 1000, since some games may have positions without evals and must be discarded
for g in games:
    g = g.copy()
    try:
        g["analysis"] = pgn_to_df(g.pop("movetext", None))
    except Exception as e:
        continue
    # g["analysis"] = pgn_to_df(g.pop("movetext", None))

    # Remove some useless fields
    g.pop("Event", None)
    g.pop("Site", None)
    g.pop("White", None)
    g.pop("Black", None)
    g.pop("WhiteTitle", None)
    g.pop("BlackTitle", None)
    g.pop("UTCDate", None)
    g.pop("UTCTime", None)
    g.pop("Termination", None)
    g.pop("WhiteRatingDiff", None)
    g.pop("BlackRatingDiff", None)

    data.append(g)
    n += 1
    if n >= N_games:
        break

In [21]:
print(len(data))
print(data[0].keys())
print(data[0]['analysis'])

1000
dict_keys(['Result', 'WhiteElo', 'BlackElo', 'ECO', 'Opening', 'TimeControl', 'analysis'])
    ply  move   san   turn  eval  ev_loss  expScore  exp_loss  clock  blunder
0     1  c2c4    c4  white    12        6    0.5135    0.0070   60.0    False
1     2  g8f6   Nf6  black    14        2    0.4840    0.0025   60.0    False
2     3  b1c3   Nc3  white    12        2    0.5135    0.0025   58.0    False
3     4  d7d5    d5  black    26       14    0.4685    0.0180   60.0    False
4     5  d2d4    d4  white     7       19    0.5075    0.0240   57.0    False
5     6  c8g4   Bg4  black    95       88    0.2640    0.2285   59.0     True
6     7  d1b3   Qb3  white    92        3    0.7230    0.0130   55.0    False
7     8  g4c8   Bc8  black   170       78    0.0385    0.2385   55.0     True
8     9  c3d5  Nxd5  white    83       87    0.6850    0.2765   53.0     True
9    10  f6d5  Nxd5  black    87        4    0.2985    0.0165   52.0    False
10   11  c4d5  cxd5  white   127       40    0

In [22]:
# Save the data to a pickle file
import pickle
with open("lichess_data.pkl", "wb") as f:
    pickle.dump(data, f)

In [23]:
# Reading the data
with open("lichess_data.pkl", "rb") as f:
    data2 = pickle.load(f)
    print(len(data2))
    print(data2[0].keys())
    print(data2[0]['analysis'])

1000
dict_keys(['Result', 'WhiteElo', 'BlackElo', 'ECO', 'Opening', 'TimeControl', 'analysis'])
    ply  move   san   turn  eval  ev_loss  expScore  exp_loss  clock  blunder
0     1  c2c4    c4  white    12        6    0.5135    0.0070   60.0    False
1     2  g8f6   Nf6  black    14        2    0.4840    0.0025   60.0    False
2     3  b1c3   Nc3  white    12        2    0.5135    0.0025   58.0    False
3     4  d7d5    d5  black    26       14    0.4685    0.0180   60.0    False
4     5  d2d4    d4  white     7       19    0.5075    0.0240   57.0    False
5     6  c8g4   Bg4  black    95       88    0.2640    0.2285   59.0     True
6     7  d1b3   Qb3  white    92        3    0.7230    0.0130   55.0    False
7     8  g4c8   Bc8  black   170       78    0.0385    0.2385   55.0     True
8     9  c3d5  Nxd5  white    83       87    0.6850    0.2765   53.0     True
9    10  f6d5  Nxd5  black    87        4    0.2985    0.0165   52.0    False
10   11  c4d5  cxd5  white   127       40    0